Importación de paquetes


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline


importar data


In [ ]:
df = pd.read_csv("/Data_PI_regresion.csv")
df.head()

Descripcion de todo los componentes

In [ ]:
df.describe().round(1)

Obtener nombres de las características

In [ ]:
df.columns

Visualización: relaciones entre todas las variables

In [ ]:
sns.pairplot(df)

Histograma: distribución del consumo de energía

In [ ]:
df['Consumo_Energia'].plot.hist(bins=25, figsize=(8, 4))

Curva de densidad del consumo de energía

In [ ]:
df['Consumo_Energia'].plot.density()


Matriz de correlación

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
# Selecciona solo las columnas con valores numéricos

numeric_df.corr().round(4)
# Cálculo de la correlación

Mapa de calor de correlación

In [ ]:
plt.figure(figsize=(10, 7))
# Comando para graficar el heatmap (completado):
sns.heatmap(numeric_df.corr(), annot=True, fmt='.4f')
plt.show()

Preparar conjuntos de características y variable objetivo

In [ ]:
l_column = list(df.columns)          # Haciendo una lista de las columnas
len_feature = len(l_column)           # Longitud de la lista de vectores de columna
l_column

Definir variables para regresión lineal

In [ ]:
X = df[l_column[0 : len_feature - 1]]   # Variables independientes: las primeras 4
y = df[l_column[len_feature - 1]]       # Variable objetivo: Consumo_Energia
X.head()

In [ ]:
y.head()

División en conjuntos de entrenamiento y prueba

In [ ]:
from sklearn.model_selection import train_test_split
# divide datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=123
)

Importar modelo y métricas

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn import metrics

In [ ]:
lm = LinearRegression()   # Creando un objeto de Regresión Lineal 'lm'

In [ ]:
lm.fit(X_train, y_train)  # Ajustar y entrenar

In [ ]:
print("El término de intersección del modelo lineal:", lm.intercept_)

In [ ]:
print("Los coeficientes del modelo lineal:", lm.coef_)

In [ ]:
cdf = pd.DataFrame(data=lm.coef_, index=X_train.columns, columns=["Coeficientes"])
cdf

Cálculo de errores estándar y estadístico t

In [ ]:
n = X_train.shape[0]           # cantidad de datos de entrenamiento
k = X_train.shape[1]           # cantidad de variables = 4
dfn = n - k                    # grados de libertad

train_pred = lm.predict(X_train)  # obtener predicciones

# calculamos el error: R²
train_error = np.square(train_pred - y_train)  # comparamos (predicción - valor real)²
sum_error = np.sum(train_error)                # suma de errores
# ¿qué tanto se equivocó el modelo?

se = [0,0,0,0]
for i in range(k):
    r = (sum_error / dfn)
    # ¿cuánto varía el valor respecto a su promedio?
    # selecciona variable: "Temperatura"
    r = r / np.sum(np.square(X_train[list(X_train.columns)[i]] - X_train[list(X_train.columns)[i]].mean()))
    se[i] = np.sqrt(r)  # cálculo del error estándar

cdf["Standard Error"] = se
cdf["t-statistic"] = cdf["Coeficientes"] / cdf["Standard Error"]
# ¿El coeficiente es grande en comparación con la incertidumbre que tiene?
# ¿está alejado de cero respecto a su error estándar?
cdf

\Gráficos de dispersión por variable vs. Consumo de Energía


In [ ]:
l = list(cdf.index)

from matplotlib import gridspec
fig = plt.figure(figsize=(18, 10))
gs = gridspec.GridSpec(2, 2)

ax0 = plt.subplot(gs[0])
ax0.scatter(df[l[0]], df['Consumo_Energia'])
ax0.set_title(f"{l[0]} vs. Consumo_Energia", fontdict={'fontsize': 20})

ax1 = plt.subplot(gs[1])
ax1.scatter(df[l[1]], df['Consumo_Energia'])
ax1.set_title(f"{l[1]} vs. Consumo_Energia", fontdict={'fontsize': 20})

ax2 = plt.subplot(gs[2])
ax2.scatter(df[l[2]], df['Consumo_Energia'])
ax2.set_title(f"{l[2]} vs. Consumo_Energia", fontdict={'fontsize': 20})

ax3 = plt.subplot(gs[3])
ax3.scatter(df[l[3]], df['Consumo_Energia'])
ax3.set_title(f"{l[3]} vs. Consumo_Energia", fontdict={'fontsize': 20})

plt.show()

Generar predicciones

In [ ]:
predictions = lm.predict(X_test)
print("Tipo del objeto predicho:", type(predictions))
print("Tamaño del objeto predicho:", predictions.shape)

Gráfico: Consumo real vs. Consumo predicho

In [ ]:
# Diagrama de dispersión
# Meta: Datos caen en una línea recta de 45°

plt.figure(figsize=(10, 7))
plt.title("Consumo de energía real vs. el predicho", fontsize=25)
plt.xlabel("Consumo de energía real", fontsize=18)
plt.ylabel("Consumo de energía predicho", fontsize=18)
plt.scatter(y_test, predictions)
plt.show()

Histograma de residuos para verificar normalidad

In [ ]:
plt.figure(figsize=(10, 7))
plt.title("Histograma de residuos para verificar la normalidad", fontsize=25)
plt.xlabel("Residuos", fontsize=18)
plt.ylabel("Densidad del kernel", fontsize=18)

# Versión original (con advertencia de obsolescencia):
sns.distplot(y_test - predictions) # histplot

plt.show()

Gráfico de residuos vs. predicciones (Homocedasticidad)

In [ ]:
plt.figure(figsize=(10, 7))
plt.title("Valores residuales vs. predichos", fontsize=25)
plt.xlabel("Consumo de energía predicho", fontsize=18)
plt.ylabel("Residuos", fontsize=18)
plt.scatter(x=predictions, y=y_test - predictions)
plt.show()

Inicio: Datos sintéticos para Bosque Aleatorio

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Importar el método make_regression para generar muestras de datos artificiales
from sklearn.datasets import make_regression

n_samples = 100     # Número de muestras
n_features = 6      # Número de características
n_informative = 3   # Número de características informativas (las que realmente influyen)

X, y, coef = make_regression(
    n_samples=n_samples,
    n_features=n_features,
    n_informative=n_informative,
    random_state=20,      # semilla aleatoria (reproducibilidad)
    shuffle=False,        # que no se mezclen las características
    noise=20,             # ruido aleatorio
    coef=True             # devuelve los coeficientes reales
)

print(coef)

Crear DataFrame y mostrar datos

In [ ]:
# Crear tabla con variables independientes: X1, X2, ..., X6
df1 = pd.DataFrame(
    data=X,
    columns=[f'X{i+1}' for i in range(n_features)]
)

# Crear columna con variable objetivo
df2 = pd.DataFrame(data=y, columns=['y'])

# Unir ambas tablas
df = pd.concat([df1, df2], axis=1)

# Mostrar primeras 10 filas
df.head(10)

Gráficos de dispersión: cada variable vs. salida

In [ ]:
with plt.style.context('Solarize_Light2'):
    for i, col in enumerate(df.columns[:-1]):  # recorrer todas las variables excepto 'y'
        plt.figure(figsize=(6, 4))
        plt.grid(True)
        plt.xlabel(f'Feature: {col}', fontsize=12)
        plt.ylabel('Output: y', fontsize=12)
        plt.scatter(df[col], df['y'], c='red', s=50, alpha=0.6)
plt.show()

División entrenamiento / prueba

In [ ]:
from sklearn import tree
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=123
)

Crear y entrenar Árbol de Decisión

In [ ]:
# Creación del modelo
tree_model = tree.DecisionTreeRegressor(max_depth=5, random_state=10)
# max_depth = niveles máximos de decisión

tree_model.fit(X_train, y_train)


Predecir y evaluar

In [ ]:
from sklearn import metrics

# Hacemos predicciones
test_pred = tree_model.predict(X_test)

plt.scatter(x=y_test, y=test_pred)  # real vs. predicho

print("Mean square error (MSE):", metrics.mean_squared_error(y_test, test_pred))
# Error cuadrático medio

Importancia de las características

In [ ]:
print("Importancia relativa de las características: ", tree_model.feature_importances_)

with plt.style.context('dark_background'):
    plt.figure(figsize=(10, 7))
    plt.grid(True)
    plt.yticks(range(n_features+1, 1, -1), df.columns[:-1], fontsize=20)
    plt.xlabel("Importancia relativa (normalizada) de los parámetros", fontsize=15)
    plt.ylabel("Características", fontsize=20)
    plt.barh(range(n_features+1, 1, -1), width=tree_model.feature_importances_, height=0.5)
plt.show()

Regresión OLS con StatsModels — Reporte completo

In [ ]:
import statsmodels.api as sm

# Añadir término constante (intersección)
Xs = sm.add_constant(X)

# Crear y ajustar modelo de Mínimos Cuadrados Ordinarios
stat_model = sm.OLS(y, Xs)
stat_result = stat_model.fit()

# Mostrar tabla completa de resultados
print(stat_result.summary())